# 🔬 Advanced Analysis Layers

This notebook demonstrates:
- Contradiction detection between papers
- Research gap identification
- Citation network construction
- Interactive network visualization

In [ ]:
# Import required libraries
import sys
sys.path.append('../src')

from embeddings import EmbeddingManager
from analysis import ContradictionDetector, ResearchGapAnalyzer, CitationNetworkBuilder
import json
import os
from pathlib import Path

# Set API keys if not already set
# os.environ['GROQ_API_KEY'] = 'your-groq-key-here'
# os.environ['GOOGLE_API_KEY'] = 'your-google-key-here'

print("✓ Imports successful")

## 1. Load Data

Load processed papers and vector store.

In [ ]:
# Check for API keys (Groq for LLM, Google for embeddings)
if not os.getenv('GROQ_API_KEY') or not os.getenv('GOOGLE_API_KEY'):
    print("⚠️ Warning: Missing API keys!")
    print("Advanced analysis requires GROQ_API_KEY and GOOGLE_API_KEY")
    print("Get Groq key at: https://console.groq.com")
    print("Get Google key at: https://makersuite.google.com/app/apikey")
    USE_MOCK = True
else:
    USE_MOCK = False
    
    # Load papers
    with open('../data/processed_papers.json', 'r') as f:
        papers = json.load(f)
    
    print(f"✓ Loaded {len(papers)} papers")
    
    # Load vector store with Google embeddings
    embedding_manager = EmbeddingManager(
        provider="google",
        persist_directory="../chroma_db"
    )
    
    try:
        vectorstore = embedding_manager.load_vectorstore(collection_name="sci_papers")
        print("✓ Vector store loaded (Google Gemini embeddings)")
    except Exception as e:
        print(f"⚠️ Could not load vector store: {e}")
        USE_MOCK = True

## 2. Contradiction Detection

Identify contradictory findings between papers on the same topic.

In [ ]:
if not USE_MOCK:
    # Initialize contradiction detector with Groq (fast & free)
    detector = ContradictionDetector(
        vectorstore=vectorstore,
        llm_provider="groq"
    )
    
    print("✓ Contradiction detector initialized")
    
    # Define research topic
    topic = "machine learning performance"
    
    print(f"\nAnalyzing contradictions on topic: '{topic}'")
    print("This may take a minute...\n")
    
    # Detect contradictions
    contradictions = detector.detect_contradictions(
        topic=topic,
        k=10
    )
    
    # Display results
    if contradictions:
        print(f"🔴 Found {len(contradictions)} potential contradiction(s):\n")
        
        for i, contradiction in enumerate(contradictions, 1):
            print(f"{'='*60}")
            print(f"Contradiction {i}:")
            print(f"{'='*60}")
            print(f"Paper 1: {contradiction['paper1_title']}")
            print(f"Paper 2: {contradiction['paper2_title']}")
            print(f"\nAnalysis:")
            print(contradiction['explanation'])
            print()
    else:
        print("✓ No contradictions detected")
        
else:
    print("⚠️ Skipped: No API key configured")
    print("\nExample contradiction detection output:")
    print("🔴 Found 1 potential contradiction:")
    print("  Paper 1: Deep Learning Approaches...")
    print("  Paper 2: Traditional Methods Outperform...")
    print("  Analysis: CONTRADICT - Paper 1 claims deep learning achieves 95% accuracy,")
    print("           while Paper 2 reports traditional methods reach 97%...")

## 3. Research Gap Analysis

Identify gaps, limitations, and future work from papers.

In [ ]:
if not USE_MOCK:
    # Initialize gap analyzer with Groq (fast & free)
    analyzer = ResearchGapAnalyzer(
        vectorstore=vectorstore,
        llm_provider="groq"
    )
    
    print("✓ Research gap analyzer initialized")
    
    print("\nIdentifying research gaps...")
    print("This may take a few minutes...\n")
    
    # Find gaps
    gaps = analyzer.find_research_gaps(k=20)
    
    # Display results
    if gaps:
        print(f"💡 Identified {len(gaps)} research gap(s):\n")
        
        for i, gap_entry in enumerate(gaps[:10], 1):  # Show first 10
            print(f"{'='*60}")
            print(f"Gap {i}:")
            print(f"{'='*60}")
            print(f"From: {gap_entry['paper_title']}")
            print(f"\nGap: {gap_entry['gap']}")
            print()
        
        if len(gaps) > 10:
            print(f"... and {len(gaps) - 10} more gaps")
    else:
        print("✓ No specific research gaps identified")
        
else:
    print("⚠️ Skipped: No API key configured")
    print("\nExample research gap output:")
    print("💡 Identified 5 research gaps:")
    print("  1. Limited evaluation on real-world datasets")
    print("  2. Lack of comparison with recent transformer models")
    print("  3. Computational efficiency not addressed")
    print("  4. Scalability to larger datasets remains unexplored")
    print("  5. Interpretability of model decisions needs investigation")

## 4. Citation Network Construction

Build and visualize citation relationships between papers.

In [ ]:
# Citation network analysis (doesn't require API key)
if not USE_MOCK:
    # Initialize builder
    builder = CitationNetworkBuilder()
    
    print("Building citation network...")
    
    # Build network
    graph = builder.build_network(papers)
    
    # Get statistics
    stats = builder.get_statistics()
    
    print("\n📊 Citation Network Statistics:")
    print(f"  Total papers: {stats['total_papers']}")
    print(f"  Total citations: {stats['total_citations']}")
    print(f"  Avg citations per paper: {stats['avg_citations_per_paper']:.2f}")
    
    if stats['most_cited']:
        most_cited_paper = next(
            p for p in papers 
            if p['paper_id'] == stats['most_cited']
        )
        print(f"  Most cited: {most_cited_paper['metadata']['title'][:50]}...")
else:
    print("⚠️ Load papers data first")

## 5. Interactive Network Visualization

Create an interactive HTML visualization of the citation network.

In [ ]:
if not USE_MOCK:
    # Generate visualization
    output_path = "../citation_graph.html"
    
    builder.visualize_network(
        output_path=output_path,
        height="750px",
        width="100%"
    )
    
    print(f"✓ Interactive visualization saved to: {output_path}")
    print(f"✓ Open the file in a web browser to explore the network")
    
    # Check if file exists
    if Path(output_path).exists():
        print(f"✓ File size: {Path(output_path).stat().st_size / 1024:.2f} KB")
else:
    print("⚠️ Skipped: Load papers data first")

## 6. Network Analysis with NetworkX

Perform graph analysis on the citation network.

In [ ]:
if not USE_MOCK:
    import networkx as nx
    
    # Network metrics
    print("📈 Network Metrics:\n")
    
    # Density
    density = nx.density(graph)
    print(f"Network density: {density:.4f}")
    
    # Degree centrality
    if graph.number_of_nodes() > 0:
        in_degree_centrality = nx.in_degree_centrality(graph)
        out_degree_centrality = nx.out_degree_centrality(graph)
        
        # Most influential (most cited)
        most_cited_id = max(in_degree_centrality, key=in_degree_centrality.get)
        most_cited_paper = next(p for p in papers if p['paper_id'] == most_cited_id)
        
        print(f"\nMost Influential Paper (by citations):")
        print(f"  Title: {most_cited_paper['metadata']['title']}")
        print(f"  In-degree centrality: {in_degree_centrality[most_cited_id]:.4f}")
        
        # Most referencing
        most_ref_id = max(out_degree_centrality, key=out_degree_centrality.get)
        most_ref_paper = next(p for p in papers if p['paper_id'] == most_ref_id)
        
        print(f"\nMost Comprehensive Literature Review:")
        print(f"  Title: {most_ref_paper['metadata']['title']}")
        print(f"  Out-degree centrality: {out_degree_centrality[most_ref_id]:.4f}")
        
        # Connected components
        num_components = nx.number_weakly_connected_components(graph)
        print(f"\nNumber of connected components: {num_components}")
        
        if num_components > 1:
            print("  (Papers form separate citation clusters)")
        else:
            print("  (All papers are interconnected)")
    else:
        print("⚠️ Empty graph")
        
else:
    print("⚠️ Skipped: Load papers data first")

## 7. Contradiction Summary Report

Generate a comprehensive summary of all contradictions found.

In [ ]:
if not USE_MOCK and 'contradictions' in locals() and contradictions:
    print("📋 CONTRADICTION SUMMARY REPORT")
    print("="*60)
    print(f"Topic analyzed: {topic}")
    print(f"Total contradictions found: {len(contradictions)}\n")
    
    for i, c in enumerate(contradictions, 1):
        print(f"{i}. {c['paper1_title'][:40]}...")
        print(f"   vs")
        print(f"   {c['paper2_title'][:40]}...")
        print(f"   → {c['explanation'][:100]}...")
        print()
else:
    print("⚠️ No contradictions to summarize")

## 8. Research Gap Clustering

Group similar research gaps together.

In [ ]:
if not USE_MOCK and 'gaps' in locals() and gaps:
    from collections import defaultdict
    
    # Simple keyword-based clustering
    gap_clusters = defaultdict(list)
    
    keywords = ['dataset', 'model', 'evaluation', 'scalability', 'interpretability', 'efficiency']
    
    for gap in gaps:
        gap_text = gap['gap'].lower()
        assigned = False
        
        for keyword in keywords:
            if keyword in gap_text:
                gap_clusters[keyword].append(gap)
                assigned = True
                break
        
        if not assigned:
            gap_clusters['other'].append(gap)
    
    print("🎯 RESEARCH GAPS BY CATEGORY\n")
    
    for category, category_gaps in gap_clusters.items():
        if category_gaps:
            print(f"{'='*60}")
            print(f"{category.upper()} ({len(category_gaps)} gaps)")
            print(f"{'='*60}")
            for gap in category_gaps[:3]:  # Show first 3
                print(f"  • {gap['gap'][:80]}...")
            if len(category_gaps) > 3:
                print(f"  ... and {len(category_gaps) - 3} more")
            print()
else:
    print("⚠️ No gaps to cluster")

## 9. Export Analysis Results

Save analysis results to files for reporting.

In [ ]:
if not USE_MOCK:
    import json
    from datetime import datetime
    
    # Prepare export data
    analysis_results = {
        'timestamp': datetime.now().isoformat(),
        'topic_analyzed': topic if 'topic' in locals() else 'N/A',
        'num_papers': len(papers),
        'contradictions': [
            {
                'paper1': c['paper1_title'],
                'paper2': c['paper2_title'],
                'explanation': c['explanation']
            }
            for c in contradictions
        ] if 'contradictions' in locals() else [],
        'research_gaps': [
            {
                'gap': g['gap'],
                'paper': g['paper_title']
            }
            for g in gaps
        ] if 'gaps' in locals() else [],
        'citation_stats': stats if 'stats' in locals() else {}
    }
    
    # Save to JSON
    output_file = '../analysis_results.json'
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(analysis_results, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Analysis results saved to: {output_file}")
    print(f"✓ Contradictions: {len(analysis_results['contradictions'])}")
    print(f"✓ Research gaps: {len(analysis_results['research_gaps'])}")
else:
    print("⚠️ Skipped: Run analysis first")

## Summary

✅ Contradiction detection completed  
✅ Research gaps identified  
✅ Citation network built and visualized  
✅ Network metrics calculated  
✅ Results exported for reporting  

⏭️ Next: Explore `05_streamlit_app.ipynb` for UI deployment